# Example 2 — Full Pipeline with Google Earth Engine

`run_crop_stage_from_gee` is the simplest entry point for GEE users: provide a
single polygon and a lookback window, and it returns the crop stage for the
last available observation.

This notebook starts with that simple one-polygon workflow. Later sections show
how to visualize the NDVI time series for that polygon, review supported input
formats, and process multiple fields in a simple batch example.

### Prerequisites

1. A GEE account — sign up at https://earthengine.google.com
2. Install both requirements files: `pip install -r requirements.txt -r requirements-gee.txt`
3. Authenticate once: `earthengine authenticate` in your terminal

Call `ee.Initialize()` **before** importing `gee_fetch`.

In [ ]:
import sys
sys.path.insert(0, "../src")

import ee
ee.Initialize()   # <-- must come before gee_fetch import

from gee_fetch import run_crop_stage_from_gee

## 1. Run the workflow for one polygon

Start with a simple polygon and call `run_crop_stage_from_gee`.

In [ ]:
polygon = [
    [-77.897937, 35.571295],
    [-77.898152, 35.570355],
    [-77.897400, 35.569852],
    [-77.895398, 35.569557],
    [-77.895062, 35.570803],
    [-77.897937, 35.571295],  # close the ring (optional)
]

START_DATE = "2023-03-01"
END_DATE = "2023-11-30"

result = run_crop_stage_from_gee(
    polygon,
    lookback_days=270,
    end_date=END_DATE,
)

for k, v in result.items():
    print(f"{k:20s}: {v}")

## 2. Visualize the NDVI time series

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from crop_stage import smooth_daily_interpolate_ndvi, estimate_stage_adaptive
from gee_fetch import fetch_ndvi

ndvi_df = fetch_ndvi(
    polygon,
    start_date=START_DATE,
    end_date=END_DATE,
    poly_name="example_field",
    buffer_m=-10,     # inset 10 m to avoid boundary pixels
)

if ndvi_df.empty:
    raise ValueError("No valid NDVI observations returned — polygon may be too cloudy, too small, or outside GEE coverage.")

df_smooth = smooth_daily_interpolate_ndvi(ndvi_df)
result = estimate_stage_adaptive(
    df_smooth["NDVI_smooth"].to_numpy(),
    dates=df_smooth["date"],
)

fig, ax = plt.subplots(figsize=(10, 4))

for sensor, grp in ndvi_df.groupby("sensor"):
    ax.scatter(grp["date"], grp["NDVI"], alpha=0.5, s=35, label=sensor)

ax.plot(df_smooth["date"], df_smooth["NDVI_smooth"], color="steelblue", lw=2, label="Smoothed")
ax.axhline(result["Lower_threshold"], color="orange", ls="--",
           label=f"Lower ({result['Lower_threshold']:.2f})")
ax.axhline(result["Upper_threshold"], color="green",  ls="--",
           label=f"Upper ({result['Upper_threshold']:.2f})")

if result.get("Peak_date"):
    ax.axvline(result["Peak_date"], color="purple", ls=":",
               label=f"Peak ({result['Peak_date'].strftime('%d-%b-%Y')})")

ax.set_ylim(0, 1)
ax.set_ylabel("NDVI")
ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(mdates.AutoDateLocator()))
ax.legend(loc="upper left", fontsize=8)
ax.set_title(f"Stage: {result['Stage']} — {result['Stage_description']}")
plt.tight_layout()
plt.show()

## 3. Supported polygon formats

`run_crop_stage_from_gee` accepts the same polygon formats as `fetch_ndvi`,
including coordinate lists, GeoJSON dictionaries, shapely geometries, GeoDataFrames,
and file paths.

In [ ]:
import geopandas as gpd
import shapely.geometry

polygon_as_list = [
    [-77.897937, 35.571295],
    [-77.898152, 35.570355],
    [-77.897400, 35.569852],
    [-77.895398, 35.569557],
    [-77.895062, 35.570803],
    [-77.897937, 35.571295],
]

polygon_as_geojson = {
    "type": "Feature",
    "geometry": {
        "type": "Polygon",
        "coordinates": [[
            [-77.897937, 35.571295],
            [-77.898152, 35.570355],
            [-77.897400, 35.569852],
            [-77.895398, 35.569557],
            [-77.895062, 35.570803],
            [-77.897937, 35.571295],
        ]]
    },
    "properties": {}
}

polygon_as_shapely = shapely.geometry.Polygon(polygon_as_list)
polygon_as_gdf = gpd.GeoDataFrame(geometry=[polygon_as_shapely], crs="EPSG:4326")
polygon_as_file = "../sample_data/sample_field.geojson"

print(type(polygon_as_list).__name__)
print(type(polygon_as_geojson).__name__)
print(type(polygon_as_shapely).__name__)
print(type(polygon_as_gdf).__name__)
print(polygon_as_file)

## 4. Batch processing

A simple batch example: run the same workflow on two nearby fields in a loop.

In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.affinity import translate
from shapely.geometry import shape
import json

with open("../sample_data/sample_field.geojson") as f:
    geom1 = shape(json.load(f)["features"][0]["geometry"])
geom2 = translate(geom1, xoff=0.05)

gdf = gpd.GeoDataFrame(
    {"field_id": ["field_a", "field_b"]},
    geometry=[geom1, geom2],
    crs="EPSG:4326",
)

field_results = []
for _, row in gdf.iterrows():
    field_gdf = gpd.GeoDataFrame([row], geometry="geometry", crs=gdf.crs)
    result = run_crop_stage_from_gee(field_gdf, lookback_days=270, end_date=END_DATE)
    field_results.append({"field_id": row["field_id"], **result})

pd.DataFrame(field_results)[["field_id", "Stage", "Stage_description", "Peak_date", "Days_since_peak"]]